# 🛣️ 02_osmnx_network_download.ipynb
================================================================================

**Script 02 of 08** in the Networks Paper analysis pipeline.

## 🎯 Purpose
Download road network data from OpenStreetMap for each Census-designated place (CDP). This creates the road network graphs that will be analyzed for egress capacity.

## 🔥 Why This Matters for Wildfire Research
The road network IS the evacuation infrastructure. By downloading the complete road network for each community, we can later analyze:
- **How many roads** lead out of the community (boundary crossings)
- **What types of roads** (highways vs. local streets)
- **Network connectivity** (are there alternative routes?)

## 🗺️ What is OSMnx?
OSMnx is a Python package that downloads and analyzes street networks from OpenStreetMap. It creates a graph data structure where:
- **Nodes** = intersections and dead ends
- **Edges** = road segments connecting nodes
- **Attributes** = road type, lanes, speed limit, name, etc.

## 📋 Workflow
1. **Load CDP shapefiles** from step 01 output
2. **For each place:**
   - Buffer the polygon (6000m to capture surrounding roads)
   - Query OpenStreetMap via bounding box
   - Download the drive network (roads cars can use)
   - Save as GraphML file for later analysis
3. **Process in parallel** for efficiency (10 workers default)

---

## 📥 INPUTS (Required Data Sources)
| Source | Path | Description |
|--------|------|-------------|
| CDP Shapefiles | `.../us-census-designated-places/{State}_{FIPS}/tl_2023_{FIPS}_place.shp` | Place boundaries (from step 01) |

## 📤 OUTPUTS (Generated Files)
| Output | Path | Description |
|--------|------|-------------|
| GraphML Files | `.../output_maps/{State}_{FIPS}/{Place}_{GEOID}/{Place}_{GEOID}.graphml` | Road network graphs |
| Log File | `.../output_maps/graph_download.log` | Processing status and errors |

## ⏱️ Expected Runtime
- ~2-4 hours for all CDPs (depends on OpenStreetMap API)
- Can be resumed if interrupted (skips existing files)

---

## ⚙️ Configuration Settings

This code automates downloading road network data for various places defined in shapefiles. Here’s the general workflow:

1. **Setup:** Specify states to process, input/output directories, and buffer size. Logging is configured, and directories are created if needed.

2. **Input Gathering:** The script looks in the input directory for state folders and shapefiles. If a state filter is defined, only those states are considered.

3. **Processing Each Place:** For each place in the shapefile:
   - A buffer is applied around the place’s polygon.
   - The polygon is used to request OSM road network data.
   - The resulting network is saved as a GraphML file in a structured output folder.

4. **Parallel Execution & Logging:** Multiple places are processed in parallel to speed things up. Progress and errors are logged, and completed tasks are summarized at the end.

In [ ]:
import os
import io
import time
import geopandas as gpd
import osmnx as ox
from shapely.geometry import box
from tqdm import tqdm
from time import perf_counter
import concurrent.futures
import logging
import networkx as nx
import multiprocessing
from datetime import datetime
import pytz

################################################################################
#                            🔧 SETTINGS & CONFIGURATION                       #
################################################################################

# =============================================================================
# ⏰ Timezone Configuration (Pacific Time for timestamps)
# =============================================================================
TIMEZONE = pytz.timezone('America/Los_Angeles')

def print_timestamped(message):
    """Print message with Pacific Time timestamp."""
    timestamp = datetime.now(TIMEZONE).strftime('%Y-%m-%d %H:%M:%S %Z')
    print(f"[{timestamp}] {message}")

# =============================================================================
# 🔧 PROCESSING SETTINGS
# =============================================================================
BUFFER_SIZE_METERS = 6000           # Buffer size in meters for network download
NUM_WORKERS = 10                    # Number of parallel workers (reduce if hitting API limits)
BATCH_SIZE = 500                    # Batch size for Overpass queries
CDP_LIMIT = None                    # Limit CDPs per shapefile (None = process all)

# =============================================================================
# 🎯 STATE & PLACE FILTERING
# =============================================================================
DOWNLOAD_ONLY_STATES = [""]         # Filter to specific states (empty = all states)
                                    # Example: ["California"] or ["California", "Texas"]
TARGET_PLACE_NAME = None            # Process only a specific place by NAME (None = all places)
                                    # Example: "Paradise" or "Macomb"

# =============================================================================
# 📝 OVERWRITE SETTINGS
# =============================================================================
OVERWRITE_EXISTING = True           # Overwrite existing GraphML files (True/False)
OVERWRITE_TODAY = False             # If False, skip files modified in last 24 hours

################################################################################
#                            📂 FILE PATHS                                      #
################################################################################
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  Configure BASE_DATA_DIR to point to your root data directory.          │
# │  You can also override it at runtime with an environment variable:      │
# │      export NETWORKS_DATA_DIR="/your/path/to/data"                      │
# │  Or edit the default path below.                                        │
# └─────────────────────────────────────────────────────────────────────────┘
BASE_DATA_DIR = os.environ.get("NETWORKS_DATA_DIR", os.path.expanduser("~/data/networks_paper"))

# =============================================================================
# 📥 INPUT PATHS
# =============================================================================
INPUT_DIR = os.path.join(BASE_DATA_DIR, 'us-census-designated-places')

# =============================================================================
# 📤 OUTPUT PATHS
# =============================================================================
OUTPUT_DIR = os.path.join(BASE_DATA_DIR, 'output_maps')
LOG_FILE = os.path.join(OUTPUT_DIR, 'graph_download.log')

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

################################################################################
#                            📝 LOGGING CONFIGURATION                          #
################################################################################

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format='%(asctime)s:%(levelname)s:%(message)s'
)

# Print current OSMnx version for reference
print_timestamped(f"📦 OSMnx version: {ox.__version__}")

################################################################################
#                            🛠️ HELPER FUNCTIONS                               #
################################################################################

def sanitize_filename(name):
    return "".join(c if c.isalnum() else "_" for c in name).strip("_")

def batch_node_ids(node_ids, batch_size=BATCH_SIZE):
    """Yield batches of node IDs to keep queries smaller."""
    for i in range(0, len(node_ids), batch_size):
        yield node_ids[i:i + batch_size]

def get_allowed_node_ids(G, allowed_types):
    """
    Return a list of node IDs from graph G that are on edges whose 'highway' attribute
    is one of the allowed_types.
    """
    allowed_nodes = set()
    for u, v, data in G.edges(data=True):
        highway = data.get("highway", None)
        if highway:
            if isinstance(highway, list):
                if any(ht in allowed_types for ht in highway):
                    allowed_nodes.add(u)
                    allowed_nodes.add(v)
            else:
                if highway in allowed_types:
                    allowed_nodes.add(u)
                    allowed_nodes.add(v)
    return list(allowed_nodes)

def safe_global_overpass_request(query, retries=3, backoff_factor=2):
    """
    A simplified Overpass request function without global delay.
    Retries the request a few times if it fails.
    """
    attempt = 0
    while attempt < retries:
        try:
            response = ox.downloader.overpass_request(query)
            return response
        except Exception as e:
            logging.error(f"Overpass request failed on attempt {attempt+1} with error: {e}")
            attempt += 1
            # Removed delay functionality; simply retry immediately
    raise Exception("Overpass request failed after multiple retries.")

def query_neighbor_graph(node_ids_batch):
    """Query Overpass for a batch of node IDs and return the corresponding graph."""
    nodes_str = ",".join(map(str, node_ids_batch))
    custom_query = f"""
    [out:xml][timeout:25];
    (
      way(bn:{nodes_str});
    );
    (._;>;);
    out body;
    """
    response_xml = safe_global_overpass_request(custom_query)
    xml_buffer = io.BytesIO(response_xml.encode('utf-8')) if isinstance(response_xml, str) else response_xml
    return ox.graph_from_xml(xml_buffer, simplify=True)

def process_place(place, state_name):
    """
    Process a single place by buffering the polygon, performing the initial OSMnx
    bounding box query, saving the resulting graph, and recording timing for each step.
    Returns a dictionary with timing information.
    """
    timings = {}
    try:
        start_total = perf_counter()
        
        place_name = place['NAME']  # Adjust if needed
        geoid = place['GEOID']      # Adjust if needed
        polygon = place.geometry

        # Setup output directory for this place
        sanitized_name = sanitize_filename(place_name)
        folder_name = f"{sanitized_name}_{geoid}"
        place_output_dir = os.path.join(OUTPUT_DIR, state_name, folder_name)
        os.makedirs(place_output_dir, exist_ok=True)
        graph_filepath = os.path.join(place_output_dir, f"{sanitized_name}_{geoid}.graphml")

        # Check if the file exists and its age
        if os.path.exists(graph_filepath):
            file_age = time.time() - os.path.getmtime(graph_filepath)
            # If OVERWRITE_TODAY is False and the file is less than 24 hours old, skip processing.
            if not OVERWRITE_TODAY and file_age < 86400:
                logging.info(f"GraphML already exists for {place_name} (GEOID: {geoid}) and was created within the last 24 hours. Skipping.")
                return timings
            elif not OVERWRITE_EXISTING:
                logging.info(f"GraphML already exists for {place_name} (GEOID: {geoid}). Skipping due to OVERWRITE_EXISTING=False.")
                return timings

        # ----------------------
        # Sub-step 1: Buffering the polygon
        # ----------------------
        start_buffer = perf_counter()
        polygon_series = gpd.GeoSeries([polygon], crs='EPSG:4326')
        utm_crs = polygon_series.estimate_utm_crs()
        buffered_polygon = polygon_series.to_crs(utm_crs).buffer(BUFFER_SIZE_METERS).to_crs('EPSG:4326').iloc[0]
        if not buffered_polygon.is_valid:
            buffered_polygon = buffered_polygon.buffer(0)
        end_buffer = perf_counter()
        timings['buffering'] = end_buffer - start_buffer

        # Create bounding box from the buffered polygon
        minx, miny, maxx, maxy = buffered_polygon.bounds
        bbox = (maxy, miny, maxx, minx)  # bbox is (north, south, east, west)

        # ----------------------
        # Sub-step 2: Initial Query using bbox
        # ----------------------
        start_initial = perf_counter()
        G_initial = ox.graph_from_bbox(
            bbox=bbox,
            network_type='drive',
            retain_all=True,
            simplify=True
        )
        end_initial = perf_counter()
        timings['initial_query'] = end_initial - start_initial

        # ----------------------
        # Sub-step 3: Saving the graph
        # ----------------------
        start_saving = perf_counter()
        ox.save_graphml(G_initial, graph_filepath)
        end_saving = perf_counter()
        timings['saving'] = end_saving - start_saving

        end_total = perf_counter()
        timings['total'] = end_total - start_total

        logging.info(f"GraphML saved for {place_name} (GEOID: {geoid}) with timings: {timings}")
        return timings
    except Exception as e:
        error_message = f"Error processing {place.get('NAME', 'Unknown')} (GEOID: {place.get('GEOID', 'Unknown')}): {e}"
        print(error_message)
        logging.error(error_message)
        return {"error": error_message}

def process_shapefile(shapefile_path, state_name, file_number, total_files):
    """
    Process one shapefile and aggregate timing data from processing each place.
    After processing all places for a state, report the average time for each sub-step.
    """
    aggregated_timings = {
        'buffering': [],
        'initial_query': [],
        'saving': [],
        'total': []
    }
    try:
        print("===== Processing shapefile start =====")
        print(f"Processing file {file_number} out of {total_files}: {os.path.basename(shapefile_path)}")
        logging.info(f"Processing file {file_number} out of {total_files}: {shapefile_path}")

        gdf_places = gpd.read_file(shapefile_path)
        logging.info(f"Shapefile {shapefile_path} loaded successfully.")

        print("Below is the head of the shapefile:")
        print(gdf_places.head())

        # Apply target location filter if specified.
        if TARGET_PLACE_NAME is not None:
            print(f"Filtering for target place: {TARGET_PLACE_NAME}")
            gdf_places = gdf_places[gdf_places['NAME'] == TARGET_PLACE_NAME]
            if gdf_places.empty:
                msg = f"No location found with name '{TARGET_PLACE_NAME}' in {os.path.basename(shapefile_path)}."
                print(msg)
                logging.info(msg)
                print("===== Processing shapefile complete =====\n")
                return aggregated_timings
        else:
            # Filter places based on population if no target is provided.
            print("Filtering places with Total Popu <= 50000.")
            initial_count = len(gdf_places)
            gdf_places = gdf_places[gdf_places['Total Popu'] <= 50000]
            filtered_count = len(gdf_places)
            print(f"Filtered out {initial_count - filtered_count} places. {filtered_count} remain.")

        # If a limit is set, slice the list of places
        places = list(gdf_places.iterrows())
        if CDP_LIMIT is not None:
            places = places[:CDP_LIMIT]
            print(f"Processing only {CDP_LIMIT} places from this shapefile.")

        if len(places) == 0:
            print("No places to process. Skipping this file.")
            logging.info("No places to process for this shapefile.")
            print("===== Processing shapefile complete =====\n")
            return aggregated_timings

        if gdf_places.crs is None:
            print("CRS is missing, setting to EPSG:4326.")
            gdf_places.set_crs(epsg=4326, inplace=True)
            logging.info(f"CRS set to EPSG:4326 for {shapefile_path}.")
        else:
            print("Reprojecting CRS to EPSG:4326.")
            gdf_places = gdf_places.to_crs(epsg=4326)
            logging.info(f"CRS reprojected to EPSG:4326 for {shapefile_path}.")

        total_places = len(places)
        successful = 0
        failed = 0
        errors = []

        print("Submitting tasks to ProcessPoolExecutor.")
        with concurrent.futures.ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
            futures = {executor.submit(process_place, place[1], state_name): place[0] for place in places}

            print("Waiting for tasks to complete.")
            for future in tqdm(concurrent.futures.as_completed(futures), total=total_places, desc=f"Processing {os.path.basename(shapefile_path)}"):
                try:
                    result = future.result()
                    if isinstance(result, dict) and result.get("error"):
                        failed += 1
                        errors.append(result["error"])
                    else:
                        successful += 1
                        # Aggregate each sub-step's timing
                        for key in aggregated_timings:
                            if key in result:
                                aggregated_timings[key].append(result[key])
                except Exception as e:
                    failed += 1
                    error_message = f"Unhandled error in a worker: {e}"
                    errors.append(error_message)
                    print(error_message)
                    logging.error(error_message)

        print(f"Processing complete for {shapefile_path}: {successful} succeeded, {failed} failed.")
        logging.info(f"Processing complete for {shapefile_path}: {successful} succeeded, {failed} failed.")
        if errors:
            print("Notifications for errors/incomplete processes:")
            for err in errors:
                print(err)
        else:
            print("All tasks completed successfully without errors.")

        # ----------------------
        # Reporting average timings per sub-step for this state (shapefile)
        # ----------------------
        print("\nAverage timings for shapefile {}:".format(os.path.basename(shapefile_path)))
        for key, times in aggregated_timings.items():
            if times:
                avg_time = sum(times) / len(times)
                print(f"  {key}: {avg_time:.2f} seconds (across {len(times)} places)")
            else:
                print(f"  {key}: No data")
        print("===== Processing shapefile complete =====\n")
        return aggregated_timings
    except Exception as e:
        logging.error(f"Error processing shapefile {shapefile_path}: {e}")
        print(f"Error processing shapefile {shapefile_path}: {e}")
        return aggregated_timings

def main():
    overall_aggregated = {
        'buffering': [],
        'initial_query': [],
        'saving': [],
        'total': []
    }
    print("===== Main execution start =====")
    shapefiles = []
    print("Listing state directories.")
    all_dirs = [d for d in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, d))]

    if DOWNLOAD_ONLY_STATES:
        missing_states = []
        for state in DOWNLOAD_ONLY_STATES:
            found = any(state.lower() in d.lower() for d in all_dirs)
            if not found:
                missing_states.append(state)
        if missing_states:
            error_msg = f"Specified state(s) {missing_states} not found in the input directory."
            print(error_msg)
            logging.error(error_msg)
            print("===== Main execution complete with errors =====")
            return

    for state_dir in all_dirs:
        if DOWNLOAD_ONLY_STATES and not any(state.lower() in state_dir.lower() for state in DOWNLOAD_ONLY_STATES):
            continue
        print(f"Listing shapefiles in state directory: {state_dir}")
        state_path = os.path.join(INPUT_DIR, state_dir)
        for file in os.listdir(state_path):
            if file.endswith('.shp'):
                shapefiles.append(os.path.join(state_path, file))

    total_files = len(shapefiles)
    if DOWNLOAD_ONLY_STATES:
        states_formatted = ", ".join(DOWNLOAD_ONLY_STATES)
        msg = f"Download mode: Only the following states will be processed: {states_formatted}."
        print(msg)
        logging.info(msg)
    else:
        msg = "Download mode: All states will be processed."
        print(msg)
        logging.info(msg)

    print("Processing each shapefile.")
    for i, shapefile_path in enumerate(shapefiles, start=1):
        state_name = os.path.basename(os.path.dirname(shapefile_path))
        print(f"Starting processing for state: {state_name}")
        logging.info(f"Starting processing for state: {state_name}")
        shapefile_timings = process_shapefile(shapefile_path, state_name, i, total_files)
        for key in overall_aggregated:
            overall_aggregated[key].extend(shapefile_timings.get(key, []))

    print("All shapefiles processed.")
    logging.info("All shapefiles processed.")

    # Overall report across all states
    print("\n===== Overall Timing Report =====")
    for key, times in overall_aggregated.items():
        if times:
            avg_time = sum(times) / len(times)
            print(f"Average {key}: {avg_time:.2f} seconds (across {len(times)} places)")
        else:
            print(f"Average {key}: No data")
    print("===== Main execution complete =====")

if __name__ == "__main__":
    main()
